In [1]:
# ============================================================
# 1. Install dependencies on Kaggle
# ============================================================
!pip install fastapi uvicorn pyngrok nest-asyncio transformers bitsandbytes torch accelerate sentence-transformers -q

# ============================================================
# 2. Imports
# ============================================================
import threading
import torch
import nest_asyncio
import uvicorn

from fastapi import FastAPI
from pydantic import BaseModel, Field
from pyngrok import ngrok
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)

# ============================================================
# 3. FastAPI application
# ============================================================
app = FastAPI(title="Regulatory Model Server")

# ============================================================
# 4. Hardcoded Model Configurations
# ============================================================
# Local path for Qwen (attached Kaggle dataset)
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1"

# Hugging Face Repo IDs for automatic online download
EMBEDDING_MODEL = "BAAI/bge-m3"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

# ============================================================
# 5. Load Qwen model (From Local Kaggle Folder)
# ============================================================
print(f"⏳ Loading local Qwen model from: '{MODEL_PATH}'...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True, # Uses local Kaggle files only
)

bnb = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("✅ Qwen Model loaded successfully onto GPU!")

# ============================================================
# 6. Load embedding + reranker models (Auto-Downloads Online)
# ============================================================
print("⏳ Downloading and loading embedding and reranker models onto CPU...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cpu", # Loaded on CPU to save GPU RAM for Qwen
)

reranker = CrossEncoder(
    RERANKER_MODEL,
    device="cpu", # Loaded on CPU to save GPU RAM for Qwen
)

print("✅ All models are ready!")

# ============================================================
# 7. Request schemas
# ============================================================
class GenerationRequest(BaseModel):
    prompt: str = Field(min_length=1)
    max_new_tokens: int = Field(default=4096, ge=1, le=8192)
    temperature: float = Field(default=0.3, ge=0, le=2)
    top_p: float = Field(default=0.95, gt=0, le=1)

class EmbeddingRequest(BaseModel):
    texts: list[str] = Field(min_length=1)

class RerankRequest(BaseModel):
    query: str = Field(min_length=1)
    documents: list[str] = Field(min_length=1)

# ============================================================
# 8. Health endpoint
# ============================================================
@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}

# ============================================================
# 9. Embedding endpoint
# ============================================================
@app.post("/embed")
def embed(request: EmbeddingRequest) -> dict[str, list[list[float]]]:
    vectors = embedding_model.encode(
        request.texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return {"embeddings": vectors.tolist()}

# ============================================================
# 10. Reranking endpoint
# ============================================================
@app.post("/rerank")
def rerank(request: RerankRequest) -> dict[str, list[float]]:
    pairs = [[request.query, document] for document in request.documents]
    scores = reranker.predict(pairs)
    return {"scores": [float(score) for score in scores]}

# ============================================================
# 11. Qwen generation endpoint
# ============================================================
@app.post("/generate")
def generate_endpoint(request: GenerationRequest) -> dict[str, str]:
    messages = [
        {
            "role": "system",
            "content": "You are a meticulous FinTech architectural compliance engine.",
        },
        {
            "role": "user",
            "content": request.prompt,
        },
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    ).to(model.device)

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = {
        **inputs,
        "streamer": streamer,
        "max_new_tokens": request.max_new_tokens,
        "temperature": request.temperature,
        "top_p": request.top_p,
        "do_sample": True,
        "repetition_penalty": 1.05,
    }

    thread = threading.Thread(
        target=model.generate,
        kwargs=generation_kwargs,
    )
    thread.start()

    full_response = ""
    for text in streamer:
        full_response += text
    thread.join()

    return {
        "status": "success",
        "response": full_response,
    }

# ============================================================
# 12. Configure ngrok (Hardcoded Token)
# ============================================================
# ⚠️ PASTE YOUR NGROK TOKEN HERE
NGROK_TOKEN = "3JGfV1r6M2pkOiGNp5hZeXdcL1q_5VzYpCD4wAM2y9n5qGwmR"

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)

print(f"\n🚀 KAGGLE LLM API URL: {public_url.public_url}/generate\n")

# ============================================================
# 13. Start Uvicorn in background
# ============================================================
def start_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
    )

server_thread = threading.Thread(
    target=start_server,
    daemon=True,
)
server_thread.start()

print("✅ Server is running in the background!")

⏳ Loading local Qwen model from: '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1'...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ Qwen Model loaded successfully onto GPU!
⏳ Downloading and loading embedding and reranker models onto CPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

... (16 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅ All models are ready!

🚀 KAGGLE LLM API URL: https://simplify-imaginary-hardness.ngrok-free.dev/generate

✅ Server is running in the background!


INFO:     Started server process [234]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
!pip install faiss-cpu -q

In [ ]:
!pip install rank-bm25 -q

In [ ]:
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr
!pip install "unstructured[pdf]" unstructured-inference

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import glob
import torch
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

In [ ]:
# =====================================================================
# 1. DOCUMENT MAPPING & PARSING (unstructured.io)
# =====================================================================
# Friendly display names mapped to your exact filenames for clean citations
# Exact dictionary map matching your os.listdir() outputs
DOC_NAME_MAP = {
    "2-API_Specifications_v2.0.0.pdf": 
        "ReBIT API Specifications v2.0.0",
        
    "3Clarification_on_Mandatory_and_Optional_fields_V1.0.pdf": 
        "ReBIT Clarification on Mandatory/Optional Fields",
        
    "4-Guidelines_Use_of_PAN_DOB_Account_Discovery.pdf": 
        "ReBIT Guidelines - PAN/DOB Account Discovery",
        
    "5-Guidelines_Adoption_Deposit_RD_TD_FI_Type_Schema_Version_2.0.0.pdf": 
        "ReBIT Deposit/RD/TD Schema Guidelines v2.0.0",
        
    "Design-Guidelines-1.pdf": 
        "Sahamati Customer Consent Design Guidelines",
        
    "Framework-for-recognising-SROs-for-the-AA-Ecosystem-1.pdf": 
        "Sahamati SRO Recognition Framework",
        
    "Master Direction - NBFC-AA Directions 2016.pdf": 
        "RBI Master Direction - NBFC-AA Directions 2016",
        
    "MeitY-Consent-Tech-Framework v1.1.pdf": 
        "MeitY Electronic Consent Framework v1.1",
        
    "Reserve Bank of India (NBFC  Account Aggregator) Directions 2025.pdf": 
        "RBI NBFC-AA Directions 2025"
}

def resolve_doc_title(filename):
    """
    Direct exact-key lookup into DOC_NAME_MAP with a safe fallback.
    """
    # 1. Try direct exact match
    if filename in DOC_NAME_MAP:
        return DOC_NAME_MAP[filename]
    
    # 2. Try matching without path prefixes (if filename contains folder paths)
    clean_base = os.path.basename(filename)
    if clean_base in DOC_NAME_MAP:
        return DOC_NAME_MAP[clean_base]
        
    # 3. Fallback: clean up raw string if new unknown PDF is added
    return clean_base.replace(".pdf", "").replace("_", " ").strip()
    
def parse_pdf_directory(folder_path="./"):
    """
    Scans folder for all PDF files, parses them using unstructured.io, 
    and extracts chunks with metadata.
    """
    # 1. Safely list all PDF files inside folder_path regardless of case extension
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Directory '{folder_path}' does not exist.")
        
    all_files = os.listdir(folder_path)
    pdf_files = [
        os.path.join(folder_path, f) 
        for f in all_files 
        if f.lower().endswith('.pdf')
    ]
    
    print(f"Found {len(pdf_files)} PDF documents in '{folder_path}' to ingest.")
    
    all_chunks = []
    
    for pdf_path in pdf_files:
        filename = os.path.basename(pdf_path)
        clean_doc_name = resolve_doc_title(filename)
        
        print(f"\n[Parsing] {filename} -> '{clean_doc_name}'...")
        try:
            # 1. Layout-aware partitioning
            elements = partition_pdf(
                filename=pdf_path,
                strategy="hi_res",
                infer_table_structure=True, # Crucial for ReBIT API tables
                extract_images_in_pdf=False
            )
            
            # 2. Intelligent Title-based chunking
            chunks = chunk_by_title(
                elements,
                multipage_sections=True,
                combine_text_under_n_chars=500,
                new_after_n_chars=1500,
                max_characters=2000
            )
            
            # 3. Format into standardized dictionary structure
            for chunk in chunks:
                page_nums = chunk.metadata.page_number
                if isinstance(page_nums, list) and len(page_nums) > 0:
                    page_str = f"Page {page_nums[0]}" if len(page_nums) == 1 else f"Pages {page_nums[0]}-{page_nums[-1]}"
                elif page_nums:
                    page_str = f"Page {page_nums}"
                else:
                    page_str = "General"
                
                all_chunks.append({
                    "text": chunk.text,
                    "meta": {
                        "doc": clean_doc_name,
                        "section": page_str
                    }
                })
                
            print(f"  └─ Extracted {len(chunks)} chunks.")
            
        except Exception as e:
            print(f"  └─ Error parsing {filename}: {e}")
            
    print(f"\nTotal ingested chunks across all documents: {len(all_chunks)}")
    return all_chunks

In [ ]:
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
)

In [ ]:
embedding_model = SentenceTransformer('BAAI/bge-m3', device = 'cpu')
cross_encoder =  CrossEncoder('BAAI/bge-reranker-v2-m3', device = 'cpu')

In [ ]:
# =====================================================================
# 2. HYBRID RAG PIPELINE (Dense + BM25 + RRF + Reranker)
# =====================================================================

class RegulatoryRAGPipeline:
    def __init__(self):
        print("\n[RAG Setup] Loading Embedding Model (BGE-M3)...")
        self.embedding_model = embedding_model
        
        print("[RAG Setup] Loading Reranker Model (BGE-Reranker-v2-M3)...")
        self.reranker = cross_encoder
        
        self.chunks = []
        self.bm25 = None
        self.faiss_index = None

    def ingest_documents(self, parsed_chunks):
        self.chunks = parsed_chunks
        
        # Sparse BM25 Index
        print("  ├─ Building BM25 index...")
        tokenized_corpus = [chunk["text"].lower().split() for chunk in self.chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)
        
        # Dense FAISS Index
        print("  ├─ Generating embeddings & building FAISS index...")
        embeddings = self.embedding_model.encode(
            [c["text"] for c in self.chunks], 
            show_progress_bar=True, 
            convert_to_numpy=True
        )
        
        faiss.normalize_L2(embeddings) # Cosine similarity
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)
        self.faiss_index.add(embeddings)
        print("  └─ Ingestion complete.")

    def retrieve(self, query, top_k_fusion=20, final_k=5):
        # A. BM25 Search
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k_fusion]
        
        # B. FAISS Dense Search
        query_emb = self.embedding_model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        dense_scores, dense_indices = self.faiss_index.search(query_emb, top_k_fusion)
        dense_top_indices = dense_indices[0]
        
        # C. Reciprocal Rank Fusion (RRF)
        rrf_scores = {idx: 0.0 for idx in set(bm25_top_indices).union(set(dense_top_indices))}
        k_const = 60
        
        for rank, idx in enumerate(bm25_top_indices):
            rrf_scores[idx] += 1.0 / (k_const + rank + 1)
        for rank, idx in enumerate(dense_top_indices):
            rrf_scores[idx] += 1.0 / (k_const + rank + 1)
            
        fused_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k_fusion]
        
        # D. Cross-Encoder Reranking
        candidates = [self.chunks[idx]["text"] for idx in fused_indices]
        rerank_pairs = [[query, doc] for doc in candidates]
        rerank_scores = self.reranker.predict(rerank_pairs)
        
        reranked_results = sorted(zip(fused_indices, rerank_scores), key=lambda x: x[1], reverse=True)
        return [self.chunks[idx] for idx, score in reranked_results[:final_k]]
        
    def generate_requirement(
        self,
        minimal_req,
        scale="Medium",
        use_rag=True,
        llm_callable=None,
        **llm_kwargs,
    ):
        if use_rag:
            retrieved_docs = self.retrieve(minimal_req)
            
            context_blocks = []
            for doc in retrieved_docs:
                meta = doc["meta"]
                block = f"[DOCUMENT: {meta['doc']} | LOCATION: {meta['section']}]\n{doc['text']}"
                context_blocks.append(block)
                
            context_str = "\n\n".join(context_blocks)
            
            prompt = f"""You are a Principal FinTech Product Manager and RBI/ReBIT Compliance Architect.
            Transform the given Minimal Requirement into a structured Software Requirement Specification (SRS/PRD).
            Target Organization Scale: {scale}
            
            STRICT COMPLIANCE & EXPLAINABILITY RULES:
            1. Base all technical, data, and security rules STRICTLY on the provided Context.
            2. Every Functional, Non-Functional, Data, and Compliance item MUST include inline citation tags (e.g., [RBI NBFC-AA Directions 2025 | Page 12]).
            3. If a section is outside the provided context, clearly note it.
            
            --- REGULATORY CONTEXT ---
            {context_str}
            
            --- MINIMAL REQUIREMENT ---
            {minimal_req}
            
            --- REFINED SRS/PRD SPECIFICATION ---
            Generate a fully structured document containing:
            1. Target Organization Scale Adjustments
            2. Stakeholders & User Roles Table
            3. Functional Requirements (with citations)
            4. Scope Boundaries (Explicit In-Scope vs Out-of-Scope)
            5. User Stories & Acceptance Criteria (Gherkin format)
            6. Non-Functional Requirements (Performance, Security, Scalability)
            7. Data & ReBIT API Requirements (Data sensitivity, retention, payloads)
            8. Compliance & Consent Governance Matrix
            9. MoSCoW Prioritization
            10. Traceability & Explainability Matrix (Mapping rules to source citations)
            """
        else:
            prompt = f"""You are a FinTech Product Manager.
            Transform the given Minimal Requirement into an SRS/PRD (Functional, NFR, Scope, Data, Compliance, MoSCoW, Traceability).
            
            --- MINIMAL REQUIREMENT ---
            {minimal_req}
            
            --- REFINED SRS/PRD SPECIFICATION ---"""
    
        return llm_callable(prompt, **llm_kwargs)       

In [ ]:
# from transformers import TextStreamer
# streamer = TextStreamer(
#     tokenizer,
#     skip_prompt=True,
#     skip_special_tokens=True,
# )

In [ ]:
from transformers import TextIteratorStreamer
from threading import Thread

In [ ]:
# =====================================================================
# 3. LLM ENGINE (Local Kaggle Path)
# =====================================================================

class QwenReasoningEngine:
    def __init__(self, model_path="/kaggle/input/qwen3-4b-instruct"):
        """
        Loads model directly from local Kaggle directory or Kaggle dataset.
        Example paths:
        - Input dataset: "/kaggle/input/your-dataset-name/Qwen3-4B-Instruct"
        - Kaggle working directory: "/kaggle/working/models/Qwen3-4B-Instruct"
        """
        print(f"\n[LLM Setup] Loading local model from: '{model_path}'...")
        
        # Verify local path exists
        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"Model directory '{model_path}' not found! "
                "Check your Kaggle input dataset path in the right sidebar under 'Input'."
            )
            
        # 1. Load Tokenizer locally
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            local_files_only=True # Prevents Hugging Face network requests
        )
        
        # 2. Load Model locally
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=bnb,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        
        print("  └─ Local model loaded successfully onto GPU.")

    def generate(
        self,
        prompt_text,
        stream=True,
        **generate_kwargs,
    ):
        messages = [
            {
                "role": "system",
                "content": "You are a meticulous FinTech architectural compliance engine."
            },
            {
                "role": "user",
                "content": prompt_text
            }
        ]
    
        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    
        inputs = self.tokenizer(
            formatted_prompt,
            return_tensors="pt"
        ).to(self.model.device)
    
        # Default generation parameters
        default_kwargs = {
            "max_new_tokens": 512,
            "temperature": 0.2,
            "do_sample": True,
            "top_p": 0.9,
            "repetition_penalty": 1.05,
        }
    
        # Override defaults with user-supplied values
        default_kwargs.update(generate_kwargs)
    
        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )
    
        generation_kwargs = {
            **inputs,
            "streamer": streamer,
            **default_kwargs,
        }
    
        thread = Thread(
            target=self.model.generate,
            kwargs=generation_kwargs,
        )
        thread.start()
    
        full_response = ""
    
        for text in streamer:
            if stream:
                print(text, end="", flush=True)
            full_response += text
    
        if stream:
            print()
    
        return full_response

In [ ]:
DATA_FOLDER = "/kaggle/input/datasets/prady2005/rag-docs/SE_Proj1" 

In [ ]:
KAGGLE_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1" 

In [ ]:
# parsed_chunks = parse_pdf_directory(folder_path=DATA_FOLDER)

In [ ]:
# len(parsed_chunks)

In [ ]:
# import pickle

# parsed_chunks = parse_pdf_directory(folder_path=DATA_FOLDER)

# with open("parsed_chunks.pkl", "wb") as f:
#     pickle.dump(parsed_chunks, f)

In [ ]:
import pickle

with open("/kaggle/input/datasets/prady2005/parsed-chunks/parsed_chunks.pkl", "rb") as f:
    parsed_chunks = pickle.load(f)

In [ ]:
rag_pipeline = RegulatoryRAGPipeline()
rag_pipeline.ingest_documents(parsed_chunks)

In [ ]:
llm_engine = QwenReasoningEngine("/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1")

In [ ]:
sample_minimal_req = "We need to allow users to link their savings accounts and fetch financial statements using PAN and DOB."

print("\n" + "="*50)
print("RUNNING WITH RAG = ON")
print("="*50)
output_rag_on = rag_pipeline.generate_requirement(
    minimal_req=sample_minimal_req,
    scale="Medium",
    use_rag=True,
    llm_callable=llm_engine.generate,
    max_new_tokens=4096,
    temperature=0.3,
    top_p=0.95,    
)
print(output_rag_on)

In [ ]:
print("\n" + "="*50)
print("RUNNING WITH RAG = OFF (BASELINE)")
print("="*50)
output_rag_off = rag_pipeline.generate_requirement(
    minimal_req=sample_minimal_req,
    scale="Medium",
    use_rag=False,
    llm_callable=llm_engine.generate,
    max_new_tokens=4096,
    temperature=0.3,
    top_p=0.95,      
)
print(output_rag_off)

In [ ]:
mkdir proj

In [ ]:
cd proj

In [ ]:
ls

In [ ]:
%%writefile config.py

import os

# Base Directories
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# Data & Model Paths (Configured for Kaggle GPU Environment)
KAGGLE_DATA_FOLDER = "/kaggle/input/datasets/prady2005/rag-docs/SE_Proj1"
KAGGLE_PARSED_CHUNKS_PATH = "/kaggle/input/datasets/prady2005/parsed-chunks/parsed_chunks.pkl"
KAGGLE_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1"

# LLM Default Generation Parameters
DEFAULT_MAX_NEW_TOKENS = 4096
DEFAULT_TEMPERATURE = 0.2
DEFAULT_TOP_P = 0.9

# Embedding & Reranker Configuration
EMBEDDING_MODEL_NAME = 'BAAI/bge-m3'
RERANKER_MODEL_NAME = 'BAAI/bge-reranker-v2-m3'
TOP_K_FUSION = 20
FINAL_RETRIEVAL_K = 5

In [ ]:
%%writefile state.py

from typing import List, Dict, Any, Optional
from typing_extensions import TypedDict

class GraphState(TypedDict):
    """
    Shared state object passed across LangGraph nodes.
    """
    user_query: str                             # Initial user minimal requirement input
    organization_scale: str                     # Target scale (Small, Medium, Enterprise)
    retrieved_context: List[Dict[str, Any]]     # RAG retrieved document chunks with metadata
    context_str: str                            # Formatted string of retrieved context with citations
    
    # Requirement Agent Outputs
    extracted_requirements: List[Dict[str, Any]]# Structured list of extracted requirements
    missing_elements: List[str]                 # Identified missing or vague details
    is_complete: bool                           # Quality gate status flag
    
    # SDLC Agent Outputs
    sdlc_factors: Dict[str, Any]                # Derived project characteristics (volatility, risk, scale)
    sdlc_recommendation: Dict[str, Any]         # Ranked SDLC scores %, chosen model, and justification
    
    # Documentation Agent Output
    final_srs_markdown: str                     # Complete, fully formatted SRS document in Markdown
    human_approved: bool                        # Human-in-the-loop review checkpoint flag
    error_message: Optional[str]                # Error log container if any node fails

In [ ]:
%%writefile schema.py

from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Literal

class RequirementItem(BaseModel):
    req_id: str = Field(description="Unique identifier e.g. FR-AA-001, SEC-001, COMP-001")
    title: str = Field(description="Short descriptive summary of requirement")
    description: str = Field(description="Detailed requirement statement")
    category: Literal[
        "Business", "Functional", "Security", "Privacy", 
        "Regulatory", "Performance", "Data Management", "Audit"
    ] = Field(description="Multi-dimensional category classification")
    priority: Literal["Must Have", "Should Have", "Could Have", "Won't Have"] = Field(description="MoSCoW priority")
    citation: str = Field(description="Source reference tag (e.g. [RBI NBFC-AA Directions 2025 | Page 12])")
    acceptance_criteria: List[str] = Field(description="Gherkin format acceptance criteria statements")

class QualityCheckResult(BaseModel):
    is_complete: bool = Field(description="True if requirements are unambiguous and actionable")
    ambiguity_score: float = Field(description="Score between 0.0 (clear) and 1.0 (highly ambiguous)")
    missing_aspects: List[str] = Field(description="List of unaddressed security, data, or operational gaps")
    clarification_questions: List[str] = Field(description="Follow-up questions to ask user if incomplete")

class SDLCScoreItem(BaseModel):
    model_name: str = Field(description="SDLC Model name e.g. DevSecOps, Agile-V-Model Hybrid, Waterfall")
    suitability_percentage: float = Field(description="Suitability match score (0-100%)")
    justification: str = Field(description="Detailed rationale referencing project traits & compliance needs")

class SDLCRecommendation(BaseModel):
    primary_recommendation: str = Field(description="Top ranked SDLC model")
    ranked_scores: List[SDLCScoreItem] = Field(description="Ranked list of evaluated SDLC models with scores")
    custom_workflow_phases: List[Dict[str, str]] = Field(
        description="Project-specific phases, key activities, security controls, and compliance gates"
    )

class SRSDocument(BaseModel):
    project_title: str
    target_scale: str
    executive_summary: str
    stakeholder_roles: List[Dict[str, str]]
    requirements_list: List[RequirementItem]
    sdlc_strategy: SDLCRecommendation
    compliance_matrix: List[Dict[str, str]]

In [ ]:
mkdir models

In [ ]:
%%writefile models/qwen_wrapper.py

from typing import Any, List, Optional, Mapping
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from pydantic import Field
import torch

class QwenLangChainChatModel(BaseChatModel):
    """
    Custom LangChain ChatModel wrapper for locally loaded Qwen3-4B-Instruct engine.
    """
    qwen_engine: Any = Field(description="Instance of QwenReasoningEngine")
    max_new_tokens: int = Field(default=4096)
    temperature: float = Field(default=0.2)
    top_p: float = Field(default=0.9)

    @property
    def _llm_type(self) -> str:
        return "qwen-local-reasoning-engine"

    def _convert_messages_to_prompt(self, messages: List[BaseMessage]) -> str:
        formatted_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                formatted_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                formatted_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                formatted_messages.append({"role": "assistant", "content": msg.content})
            else:
                formatted_messages.append({"role": "user", "content": msg.content})
                
        formatted_prompt = self.qwen_engine.tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=True
        )
        return formatted_prompt

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        
        prompt_str = self._convert_messages_to_prompt(messages)
        
        # Override defaults if passed in kwargs
        gen_kwargs = {
            "max_new_tokens": kwargs.get("max_new_tokens", self.max_new_tokens),
            "temperature": kwargs.get("temperature", self.temperature),
            "top_p": kwargs.get("top_p", self.top_p),
            "do_sample": True if kwargs.get("temperature", self.temperature) > 0 else False
        }
        
        # Tokenize and run on model device
        inputs = self.qwen_engine.tokenizer(prompt_str, return_tensors="pt").to(self.qwen_engine.model.device)
        
        with torch.no_grad():
            output_tokens = self.qwen_engine.model.generate(
                **inputs,
                **gen_kwargs
            )
            
        # Extract response portion (strip prompt)
        input_len = inputs["input_ids"].shape[1]
        generated_tokens = output_tokens[0][input_len:]
        response_text = self.qwen_engine.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        ai_message = AIMessage(content=response_text)
        generation = ChatGeneration(message=ai_message)
        
        return ChatResult(generations=[generation])

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {
            "model_type": self._llm_type,
            "temperature": self.temperature,
            "max_new_tokens": self.max_new_tokens
        }

In [ ]:
ls models

In [ ]:
mkdir rag

In [ ]:
%%writefile rag/hybrid_retriever.py

import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from typing import List, Dict, Any
import config

class HybridRegulatoryRetriever:
    """
    4-Tier Hybrid Retrieval Pipeline:
    1. Sparse BM25 Search
    2. Dense FAISS Search (BGE-M3)
    3. Reciprocal Rank Fusion (RRF)
    4. Cross-Encoder Reranking (BGE-Reranker-v2-M3)
    """
    def __init__(self, embedding_model=None, cross_encoder=None):
        print("\n[RAG Setup] Initializing Hybrid Regulatory Retriever...")
        self.embedding_model = embedding_model if embedding_model else SentenceTransformer(
            config.EMBEDDING_MODEL_NAME, device='cpu'
        )
        self.reranker = cross_encoder if cross_encoder else CrossEncoder(
            config.RERANKER_MODEL_NAME, device='cpu'
        )
        self.chunks = []
        self.bm25 = None
        self.faiss_index = None

    def ingest_chunks(self, parsed_chunks: List[Dict[str, Any]]):
        """
        Builds sparse BM25 index and dense FAISS index from pre-parsed chunks.
        """
        self.chunks = parsed_chunks
        
        # 1. Sparse BM25 Index
        tokenized_corpus = [chunk["text"].lower().split() for chunk in self.chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)
        
        # 2. Dense FAISS Index
        embeddings = self.embedding_model.encode(
            [c["text"] for c in self.chunks], 
            show_progress_bar=False, 
            convert_to_numpy=True
        )
        faiss.normalize_L2(embeddings)
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)
        self.faiss_index.add(embeddings)
        print(f"[RAG Setup] Ingested {len(self.chunks)} chunks into FAISS & BM25 indexes.")

    def retrieve(
        self, 
        query: str, 
        top_k_fusion: int = config.TOP_K_FUSION, 
        final_k: int = config.FINAL_RETRIEVAL_K
    ) -> List[Dict[str, Any]]:
        """
        Executes hybrid search across BM25 & FAISS, fuses with RRF, and reranks via Cross-Encoder.
        """
        if not self.chunks:
            raise ValueError("No chunks ingested in retriever. Call ingest_chunks() first.")

        # A. BM25 Search
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k_fusion]
        
        # B. FAISS Dense Search
        query_emb = self.embedding_model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        dense_scores, dense_indices = self.faiss_index.search(query_emb, top_k_fusion)
        dense_top_indices = dense_indices[0]
        
        # C. Reciprocal Rank Fusion (RRF)
        rrf_scores = {idx: 0.0 for idx in set(bm25_top_indices).union(set(dense_top_indices))}
        k_const = 60
        
        for rank, idx in enumerate(bm25_top_indices):
            rrf_scores[idx] += 1.0 / (k_const + rank + 1)
        for rank, idx in enumerate(dense_top_indices):
            rrf_scores[idx] += 1.0 / (k_const + rank + 1)
            
        fused_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k_fusion]
        
        # D. Cross-Encoder Reranking
        candidates = [self.chunks[idx]["text"] for idx in fused_indices]
        rerank_pairs = [[query, doc] for doc in candidates]
        rerank_scores = self.reranker.predict(rerank_pairs)
        
        reranked_results = sorted(zip(fused_indices, rerank_scores), key=lambda x: x[1], reverse=True)
        return [self.chunks[idx] for idx, score in reranked_results[:final_k]]

    def format_context_string(self, retrieved_chunks: List[Dict[str, Any]]) -> str:
        """
        Formats retrieved chunks into clean markdown blocks with source citation tags.
        """
        context_blocks = []
        for doc in retrieved_chunks:
            meta = doc["meta"]
            block = f"[DOCUMENT: {meta['doc']} | LOCATION: {meta['section']}]\n{doc['text']}"
            context_blocks.append(block)
        return "\n\n".join(context_blocks)

In [ ]:
mkdir agents

In [ ]:
%%writefile agents/requirement_agent.py

import json
from typing import Dict, Any, List
from langchain_core.messages import SystemMessage, HumanMessage
from models.qwen_wrapper import QwenLangChainChatModel
import re

def parse_llm_json(response_text: str) -> dict:
    """Safely extracts JSON from reasoning models by stripping <think> tags."""
    # 1. Remove <think>...</think> blocks completely
    text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL).strip()
    
    # 2. Check for markdown code blocks
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
        
    # 3. Failsafe: Grab everything between the first { and the last }
    start_idx = text.find('{')
    end_idx = text.rfind('}')
    
    if start_idx != -1 and end_idx != -1:
        clean_json = text[start_idx:end_idx+1]
        return json.loads(clean_json)
    
    raise ValueError("No valid JSON found in the response.")

class RequirementExtractionAgent:
    """
    Agent responsible for extracting, categorizing, and mapping requirements to regulatory citations.
    """
    def __init__(self, llm: QwenLangChainChatModel):
        self.llm = llm

    def run(self, user_query: str, context_str: str, target_scale: str = "Medium") -> Dict[str, Any]:
        system_prompt = """You are a Principal FinTech Product Manager and RBI/ReBIT Compliance Architect.
Your task is to analyze user requirement statements against regulatory context and extract structured requirements.

CLASSIFICATION DIMENSIONS:
- Functional
- Security
- Privacy
- Regulatory
- Data Management
- Performance
- Audit

STRICT CITATION RULE:
Every extracted item MUST explicitly cite its source from the context using standard brackets, e.g., [RBI NBFC-AA Directions 2025 | Page 12].

Return your output as a valid JSON object matching this structure:
{
  "extracted_requirements": [
    {
      "req_id": "FR-AA-001",
      "title": "Short Summary",
      "description": "Detailed requirement text",
      "category": "Functional | Security | Privacy | Regulatory | Data Management | Performance | Audit",
      "priority": "Must Have | Should Have | Could Have | Won't Have",
      "citation": "[Document Name | Location]",
      "acceptance_criteria": ["Given ... When ... Then ..."]
    }
  ]
}"""

        user_prompt = f"""--- TARGET SCALE ---
{target_scale}

--- REGULATORY CONTEXT ---
{context_str}

--- USER REQUIREMENT QUERY ---
{user_query}

Extract and classify all software requirements based on the context above."""

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        response = self.llm.invoke(messages)
        
        try:
            return parse_llm_json(response.content)
        except Exception:
            print(f"  [Error] Failed to parse LLM output: {e}")
            return {
                "extracted_requirements": [
                    {
                        "req_id": "REQ-001",
                        "title": "Extracted Requirement Overview",
                        "description": response.content,
                        "category": "Functional",
                        "priority": "Must Have",
                        "citation": "General Context",
                        "acceptance_criteria": ["Scenario: Successful execution"]
                    }
                ]
            }

In [ ]:
%%writefile agents/quality_agent.py

import json
from typing import Dict, Any
from langchain_core.messages import SystemMessage, HumanMessage
from models.qwen_wrapper import QwenLangChainChatModel
import re

def parse_llm_json(response_text: str) -> dict:
    """Safely extracts JSON from reasoning models by stripping <think> tags."""
    # 1. Remove <think>...</think> blocks completely
    text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL).strip()
    
    # 2. Check for markdown code blocks
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
        
    # 3. Failsafe: Grab everything between the first { and the last }
    start_idx = text.find('{')
    end_idx = text.rfind('}')
    
    if start_idx != -1 and end_idx != -1:
        clean_json = text[start_idx:end_idx+1]
        return json.loads(clean_json)
    
    raise ValueError("No valid JSON found in the response.")

class QualityCheckAgent:
    """
    Validates requirement clarity, ambiguity, and completeness.
    """
    def __init__(self, llm: QwenLangChainChatModel):
        self.llm = llm

    def run(self, user_query: str, extracted_reqs: list) -> Dict[str, Any]:
        system_prompt = """You are a Lead Quality Assurance & FinTech Audit Specialist.
Analyze the extracted software requirements for:
1. Ambiguity or vague statements
2. Missing security, consent, or data retention controls
3. Incompleteness or missing dependencies

Return a valid JSON object matching:
{
  "is_complete": true/false,
  "ambiguity_score": 0.0 to 1.0,
  "missing_aspects": ["Gap 1", "Gap 2"],
  "clarification_questions": ["Question 1", "Question 2"]
}"""

        reqs_summary = json.dumps(extracted_reqs, indent=2)
        user_prompt = f"""--- INITIAL QUERY ---
{user_query}

--- EXTRACTED REQUIREMENTS ---
{reqs_summary}

Evaluate requirement completeness and quality."""

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        response = self.llm.invoke(messages)
        
        try:
            return parse_llm_json(response.content)
        except Exception:
            print(f"  [Error] Failed to parse LLM output: {e}")
            return {
                "is_complete": True,
                "ambiguity_score": 0.1,
                "missing_aspects": [],
                "clarification_questions": []
            }

In [ ]:
%%writefile agents/sdlc_agent.py

import json
from typing import Dict, Any
from langchain_core.messages import SystemMessage, HumanMessage
from models.qwen_wrapper import QwenLangChainChatModel
import re

def parse_llm_json(response_text: str) -> dict:
    """Safely extracts JSON from reasoning models by stripping <think> tags."""
    # 1. Remove <think>...</think> blocks completely
    text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL).strip()
    
    # 2. Check for markdown code blocks
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
        
    # 3. Failsafe: Grab everything between the first { and the last }
    start_idx = text.find('{')
    end_idx = text.rfind('}')
    
    if start_idx != -1 and end_idx != -1:
        clean_json = text[start_idx:end_idx+1]
        return json.loads(clean_json)
    
    raise ValueError("No valid JSON found in the response.")

class SDLCSelectionAgent:
    """
    Agent responsible for analyzing project traits, calculating ranked SDLC suitability scores,
    and generating customized project-specific workflows.
    """
    def __init__(self, llm: QwenLangChainChatModel):
        self.llm = llm

    def run(self, extracted_reqs: list, target_scale: str = "Medium") -> Dict[str, Any]:
        system_prompt = """You are a Senior Software Architect and SDLC Strategy Consultant for Banking Systems.
Analyze the project requirements and scale to derive key decision parameters:
- Requirement Volatility (High/Med/Low)
- Regulatory & Security Criticality (High/Med/Low)
- System Complexity & Failure Impact (High/Med/Low)

Based on these parameters, evaluate and rank the following SDLC models with suitability percentage scores (0-100%):
- DevSecOps
- Agile-V-Model Hybrid
- V-Model
- Spiral
- Waterfall

Then, generate a customized SDLC workflow specifying:
- Phases
- Activities
- Security Gates & Regulatory Checkpoints

Return a valid JSON object matching:
{
  "derived_factors": {
    "volatility": "High|Med|Low",
    "regulatory_criticality": "High|Med|Low",
    "security_risk": "High|Med|Low"
  },
  "sdlc_recommendation": {
    "primary_recommendation": "Name of Top Model",
    "ranked_scores": [
      {
        "model_name": "Model Name",
        "suitability_percentage": 88.0,
        "justification": "Detailed rationale"
      }
    ],
    "custom_workflow_phases": [
      {
        "phase_name": "Phase 1: Security & Requirement Baseline",
        "key_activities": "Activities description",
        "compliance_checkpoints": "RBI/ReBIT sign-off details"
      }
    ]
  }
}"""

        reqs_summary = json.dumps(extracted_reqs, indent=2)
        user_prompt = f"""--- TARGET SCALE ---
{target_scale}

--- EXTRACTED REQUIREMENTS ---
{reqs_summary}

Determine the optimal SDLC model and generate the project-specific workflow."""

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        response = self.llm.invoke(messages)
        
        try:
            return parse_llm_json(response.content)
        except Exception:
            print(f"  [Error] Failed to parse LLM output: {e}")
            return {
                "derived_factors": {"volatility": "Medium", "regulatory_criticality": "High", "security_risk": "High"},
                "sdlc_recommendation": {
                    "primary_recommendation": "DevSecOps",
                    "ranked_scores": [
                        {"model_name": "DevSecOps", "suitability_percentage": 90.0, "justification": "High security & continuous compliance requirements."},
                        {"model_name": "Agile-V-Model Hybrid", "suitability_percentage": 82.0, "justification": "Rigorous validation combined with iterative development."}
                    ],
                    "custom_workflow_phases": [
                        {"phase_name": "Design & Compliance Gate", "key_activities": "Threat modeling & ReBIT API schema mapping", "compliance_checkpoints": "Mandatory CISA & InfoSec approval"}
                    ]
                }
            }

In [ ]:
%%writefile agents/doc_agent.py

from typing import Dict, Any, List

class DocumentationAgent:
    """
    Assembles extracted requirements, compliance citations, and SDLC recommendations into
    a structured Software Requirements Specification (SRS).
    """
    def generate_srs_markdown(
        self, 
        user_query: str, 
        scale: str, 
        requirements: List[Dict[str, Any]], 
        sdlc_data: Dict[str, Any]
    ) -> str:
        
        sdlc_rec = sdlc_data.get("sdlc_recommendation", {})
        primary_sdlc = sdlc_rec.get("primary_recommendation", "DevSecOps")
        scores = sdlc_rec.get("ranked_scores", [])
        phases = sdlc_rec.get("custom_workflow_phases", [])

        # Build Requirements Section
        req_rows = []
        for r in requirements:
            req_rows.append(
                f"### {r.get('req_id', 'REQ')} - {r.get('title', 'Requirement')}\n"
                f"- **Category:** {r.get('category', 'Functional')}\n"
                f"- **Priority:** {r.get('priority', 'Must Have')}\n"
                f"- **Citation:** `{r.get('citation', 'N/A')}`\n"
                f"- **Description:** {r.get('description', '')}\n"
                f"- **Acceptance Criteria:**\n" + 
                "\n".join([f"  - {ac}" for ac in r.get("acceptance_criteria", [])]) + "\n"
            )
        
        reqs_text = "\n".join(req_rows)

        # Build SDLC Ranking Section
        sdlc_rows = []
        for s in scores:
            sdlc_rows.append(
                f"| {s.get('model_name')} | {s.get('suitability_percentage')}% | {s.get('justification')} |"
            )
        sdlc_table = "\n".join(sdlc_rows)

        # Build SDLC Workflow Section
        workflow_rows = []
        for idx, p in enumerate(phases, 1):
            workflow_rows.append(
                f"#### Phase {idx}: {p.get('phase_name')}\n"
                f"- **Activities:** {p.get('key_activities')}\n"
                f"- **Compliance & Security Gate:** {p.get('compliance_checkpoints')}\n"
            )
        workflow_text = "\n".join(workflow_rows)

        # Final Document Assembly
        srs_md = f"""# Software Requirements Specification (SRS) & SDLC Recommendation

## 1. Executive Summary & System Scope
- **Initial Request:** {user_query}
- **Target Organization Scale:** {scale}
- **Recommended SDLC Approach:** **{primary_sdlc}**

---

## 2. Classified Requirements & Compliance Mappings

{reqs_text}

---

## 3. SDLC Selection & Evaluation

### 3.1 Model Suitability Rankings
| SDLC Model | Suitability Score (%) | Rationale & Justification |
| :--- | :--- | :--- |
{sdlc_table}

### 3.2 Customized Project SDLC Workflow
{workflow_text}

---

## 4. Governance & Human Approval Gate
- [ ] Requirements Baseline Approved by Product Manager
- [ ] Compliance & Regulatory Citations Verified by Legal Officer
- [ ] SDLC Strategy & Security Gates Approved by Solutions Architect
"""
        return srs_md

In [ ]:
mkdir graph

In [ ]:
%%writefile graph/human_approval.py

from typing import Dict, Any
from state import GraphState

def human_approval_node(state: GraphState) -> Dict[str, Any]:
    """
    Human-in-the-loop approval checkpoint node.
    In interactive mode or stateful persistence, this node acts as an interrupt gate.
    """
    print("\n" + "="*60)
    print("      HUMAN-IN-THE-LOOP APPROVAL CHECKPOINT")
    print("="*60)
    
    extracted_reqs = state.get("extracted_requirements", [])
    sdlc_rec = state.get("sdlc_recommendation", {})
    primary_sdlc = sdlc_rec.get("primary_recommendation", "DevSecOps")
    
    print(f"\n[Review Summary]")
    print(f"- Total Extracted Requirements: {len(extracted_reqs)}")
    print(f"- Recommended SDLC Model: {primary_sdlc}")
    print("\nReviewing compliance citations and SDLC justification...")
    
    # In automated runner execution, default to True; in production with state persistence,
    # LangGraph interrupt() or human intervention sets this flag.
    return {"human_approved": True}

In [ ]:
%%writefile graph/workflow.py

from langgraph.graph import StateGraph, END
from typing import Dict, Any, Literal

from state import GraphState
from rag.hybrid_retriever import HybridRegulatoryRetriever
from agents.requirement_agent import RequirementExtractionAgent
from agents.quality_agent import QualityCheckAgent
from agents.sdlc_agent import SDLCSelectionAgent
from agents.doc_agent import DocumentationAgent
from graph.human_approval import human_approval_node

class FinTechRequirementsWorkflow:
    """
    Stateful multi-agent orchestrator built with LangGraph.
    """
    def __init__(
        self, 
        retriever: HybridRegulatoryRetriever,
        req_agent: RequirementExtractionAgent,
        quality_agent: QualityCheckAgent,
        sdlc_agent: SDLCSelectionAgent,
        doc_agent: DocumentationAgent
    ):
        self.retriever = retriever
        self.req_agent = req_agent
        self.quality_agent = quality_agent
        self.sdlc_agent = sdlc_agent
        self.doc_agent = doc_agent
        self.app = self._build_graph()

    # --- Node Handlers ---
    def retrieve_node(self, state: GraphState) -> Dict[str, Any]:
        query = state["user_query"]
        print(f"\n[Node 1: RAG Retriever] Retrieving context for: '{query}'...")
        retrieved_chunks = self.retriever.retrieve(query)
        context_str = self.retriever.format_context_string(retrieved_chunks)
        return {
            "retrieved_context": retrieved_chunks,
            "context_str": context_str
        }

    def requirement_extraction_node(self, state: GraphState) -> Dict[str, Any]:
        print("\n[Node 2: Requirement Agent] Extracting & classifying requirements...")
        query = state["user_query"]
        context_str = state["context_str"]
        scale = state.get("organization_scale", "Medium")
        
        result = self.req_agent.run(query, context_str, scale)
        reqs = result.get("extracted_requirements", [])
        print(f"  └─ Extracted {len(reqs)} structured requirements.")
        return {"extracted_requirements": reqs}

    def quality_check_node(self, state: GraphState) -> Dict[str, Any]:
        print("\n[Node 3: Quality Agent] Evaluating requirement completeness...")
        query = state["user_query"]
        reqs = state["extracted_requirements"]
        
        result = self.quality_agent.run(query, reqs)
        is_complete = result.get("is_complete", True)
        missing = result.get("missing_aspects", [])
        
        print(f"  └─ Quality Status: {'COMPLETE' if is_complete else 'INCOMPLETE / GAPS IDENTIFIED'}")
        return {
            "is_complete": is_complete,
            "missing_elements": missing
        }

    def sdlc_selection_node(self, state: GraphState) -> Dict[str, Any]:
        print("\n[Node 4: SDLC Agent] Evaluating project traits & ranking SDLC models...")
        reqs = state["extracted_requirements"]
        scale = state.get("organization_scale", "Medium")
        
        sdlc_output = self.sdlc_agent.run(reqs, scale)
        factors = sdlc_output.get("derived_factors", {})
        rec = sdlc_output.get("sdlc_recommendation", {})
        print(f"  └─ Primary SDLC Recommendation: {rec.get('primary_recommendation')}")
        return {
            "sdlc_factors": factors,
            "sdlc_recommendation": rec
        }

    def documentation_node(self, state: GraphState) -> Dict[str, Any]:
        print("\n[Node 5: Documentation Agent] Generating final Markdown SRS document...")
        query = state["user_query"]
        scale = state.get("organization_scale", "Medium")
        reqs = state["extracted_requirements"]
        sdlc_data = {
            "sdlc_factors": state.get("sdlc_factors", {}),
            "sdlc_recommendation": state.get("sdlc_recommendation", {})
        }
        
        srs_markdown = self.doc_agent.generate_srs_markdown(query, scale, reqs, sdlc_data)
        return {"final_srs_markdown": srs_markdown}

    # --- Conditional Edge Routing ---
    def route_after_quality_check(self, state: GraphState) -> Literal["sdlc_selection", "documentation"]:
        # If requirements are complete, proceed to SDLC selection; otherwise continue directly to doc generation with notes
        if state.get("is_complete", True):
            return "sdlc_selection"
        else:
            print("  └─ Routing directly forward with noted quality gaps.")
            return "sdlc_selection"

    # --- Graph Assembly ---
    def _build_graph(self):
        workflow = StateGraph(GraphState)

        # Add Nodes
        workflow.add_node("retrieve", self.retrieve_node)
        workflow.add_node("extract_requirements", self.requirement_extraction_node)
        workflow.add_node("quality_check", self.quality_check_node)
        workflow.add_node("sdlc_selection", self.sdlc_selection_node)
        workflow.add_node("human_approval", human_approval_node)
        workflow.add_node("generate_documentation", self.documentation_node)

        # Set Entry Point
        workflow.set_entry_point("retrieve")

        # Define Edges
        workflow.add_edge("retrieve", "extract_requirements")
        workflow.add_edge("extract_requirements", "quality_check")
        
        # Conditional Edge after Quality Check
        workflow.add_conditional_edges(
            "quality_check",
            self.route_after_quality_check,
            {
                "sdlc_selection": "sdlc_selection"
            }
        )
        
        workflow.add_edge("sdlc_selection", "human_approval")
        workflow.add_edge("human_approval", "generate_documentation")
        workflow.add_edge("generate_documentation", END)

        return workflow.compile()

In [ ]:
%%writefile main.py

import os
import pickle
import config

# Import Custom Modules
from models.qwen_wrapper import QwenLangChainChatModel
from rag.hybrid_retriever import HybridRegulatoryRetriever
from agents.requirement_agent import RequirementExtractionAgent
from agents.quality_agent import QualityCheckAgent
from agents.sdlc_agent import SDLCSelectionAgent
from agents.doc_agent import DocumentationAgent
from graph.workflow import FinTechRequirementsWorkflow

def run_application():
    print("="*70)
    print("  AGENTIC-AI FINTECH REQUIREMENT ENGINEERING & SDLC SYSTEM  ")
    print("="*70)

    # 1. Load Pre-parsed Regulatory Chunks
    print(f"\n[Step 1] Loading parsed chunks from '{config.KAGGLE_PARSED_CHUNKS_PATH}'...")
    if not os.path.exists(config.KAGGLE_PARSED_CHUNKS_PATH):
        raise FileNotFoundError(f"Parsed chunks file not found at {config.KAGGLE_PARSED_CHUNKS_PATH}")
        
    with open(config.KAGGLE_PARSED_CHUNKS_PATH, "rb") as f:
        parsed_chunks = pickle.load(f)
    print(f"  └─ Successfully loaded {len(parsed_chunks)} regulatory document chunks.")

    # 2. Load Local Qwen Model Engine
    print(f"\n[Step 2] Loading local Qwen model from '{config.KAGGLE_MODEL_PATH}'...")
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    import torch

    bnb = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(config.KAGGLE_MODEL_PATH, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        config.KAGGLE_MODEL_PATH,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    class QwenEngine:
        def __init__(self, tok, mdl):
            self.tokenizer = tok
            self.model = mdl

    raw_qwen_engine = QwenEngine(tokenizer, model)
    qwen_llm = QwenLangChainChatModel(
        qwen_engine=raw_qwen_engine,
        max_new_tokens=config.DEFAULT_MAX_NEW_TOKENS,
        temperature=config.DEFAULT_TEMPERATURE
    )
    print("  └─ Local Qwen ChatModel wrapper ready.")

    # 3. Build RAG Retriever
    print("\n[Step 3] Building Hybrid RAG Retriever...")
    retriever = HybridRegulatoryRetriever()
    retriever.ingest_chunks(parsed_chunks)

    # 4. Instantiate Specialized Agents
    print("\n[Step 4] Initializing Specialized AI Agents...")
    req_agent = RequirementExtractionAgent(qwen_llm)
    quality_agent = QualityCheckAgent(qwen_llm)
    sdlc_agent = SDLCSelectionAgent(qwen_llm)
    doc_agent = DocumentationAgent()

    # 5. Assemble LangGraph Workflow
    print("\n[Step 5] Assembling LangGraph Multi-Agent Workflow...")
    app_workflow = FinTechRequirementsWorkflow(
        retriever=retriever,
        req_agent=req_agent,
        quality_agent=quality_agent,
        sdlc_agent=sdlc_agent,
        doc_agent=doc_agent
    )

    # 6. Execute Application on Sample Input Query
    sample_query = (
        "We need to allow users to link their savings accounts and fetch financial statements "
        "using PAN and DOB, while strictly adhering to consent management and data security guidelines."
    )
    
    initial_state = {
        "user_query": sample_query,
        "organization_scale": "Medium",
        "retrieved_context": [],
        "context_str": "",
        "extracted_requirements": [],
        "missing_elements": [],
        "is_complete": True,
        "sdlc_factors": {},
        "sdlc_recommendation": {},
        "final_srs_markdown": "",
        "human_approved": False,
        "error_message": None
    }

    print("\n" + "="*70)
    print("  EXECUTING LANGGRAPH MULTI-AGENT PIPELINE  ")
    print("="*70)
    
    final_output = app_workflow.app.invoke(initial_state)

    print("\n" + "="*70)
    print("  FINAL GENERATED SRS & SDLC ARTIFACT  ")
    print("="*70 + "\n")
    print(final_output["final_srs_markdown"])

if __name__ == "__main__":
    run_application()

In [ ]:
ls

In [ ]:
!python main.py

In [ ]:
%%writefile requirements.txt

# ==========================================
# FinTech Requirements Engine Dependencies
# ==========================================

# 1. Agent & Graph Orchestration
langchain>=0.1.0
langchain-core>=0.1.0
langgraph>=0.0.26
pydantic>=2.0.0

# 2. Local LLM Execution (Qwen3)
torch>=2.1.2
transformers>=4.38.0
bitsandbytes>=0.46.1
accelerate>=0.27.0

# 3. Hybrid RAG Pipeline
sentence-transformers>=2.5.0
faiss-cpu>=1.7.4      # Note: Change to faiss-gpu if running outside Kaggle with a dedicated GPU
rank-bm25>=0.2.2
numpy>=1.24.0

# 4. Document Parsing & Ingestion
unstructured[pdf]>=0.12.0
unstructured-inference>=0.7.2
pdf2image>=1.17.0
pdfminer.six>=20221105

In [ ]:
hi

In [ ]:
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr

In [ ]:
!pip install -r requirements.txt